# 🔍 Módulo 11: El Motor de Búsqueda Semántica — Vector Store
## Notebook de Conocimiento Guiado

**Objetivo:** Implementar tres algoritmos de búsqueda de vecinos más cercanos (ANN)
y entender los tradeoffs entre exactitud, velocidad y memoria.

**Lore:** Eres el arquitecto de la Biblioteca Alienígena. Trillones de documentos
almacenados como vectores. Tu misión: responder consultas en microsegundos.

| Sección | Concepto |
|---------|---------|
| 📚 Teoría | Embeddings, similitud coseno, ANN |
| 🔨 Demo | Fuerza bruta exacta O(n·d) |
| 🔨 Demo | KD-Tree O(d·log n) promedio |
| 🔨 Demo | LSH O(1) aproximado |
| ✏️ Ejercicio 1 | Búsqueda por rango (radio search) |
| ✏️ Ejercicio 2 | IVF: Inverted File Index básico |
| 🎯 Quiz | FAISS, HNSW, Pinecone, Weaviate |

## 📚 Parte 1: El Problema de los Vecinos Más Cercanos

Un **embedding** convierte texto/imágenes en vectores numéricos donde
la similitud semántica corresponde a distancia geométrica pequeña.

```
"perro"     → [0.9, 0.1, 0.3, ...]
"gato"      → [0.8, 0.2, 0.3, ...]   ← cerca de "perro"
"automóvil" → [0.1, 0.8, 0.6, ...]   ← lejos de "perro"
```

**Métricas de similitud:**

| Métrica | Fórmula | Cuando usar |
|---------|---------|------------|
| Coseno | cos(θ) = a·b / (‖a‖·‖b‖) | Embeddings de texto (normaliza la magnitud) |
| Euclidiana | √Σ(aᵢ-bᵢ)² | Coordenadas geométricas |
| Producto punto | Σaᵢ·bᵢ | Cuando la magnitud es relevante |

**El problema de escala:**
- 1M documentos × 768 dimensiones = ~3GB en float32
- Búsqueda bruta: O(1M × 768) operaciones por query → demasiado lento
- Necesitamos estructuras de índice: KD-Tree, HNSW, IVF, LSH

In [ ]:
# 🔨 IMPLEMENTACIÓN: Métricas de distancia

import math, random

def producto_punto(a, b):
    return sum(x*y for x,y in zip(a,b))

def similitud_coseno(a, b):
    dot = producto_punto(a, b)
    norma_a = math.sqrt(sum(x*x for x in a))
    norma_b = math.sqrt(sum(y*y for y in b))
    if norma_a == 0 or norma_b == 0: return 0.0
    return dot / (norma_a * norma_b)

def distancia_euclidiana(a, b):
    return math.sqrt(sum((x-y)**2 for x,y in zip(a,b)))

def normalizar(v):
    norma = math.sqrt(sum(x*x for x in v))
    return [x/norma for x in v] if norma > 0 else list(v)

# Demo con vectores 3D para visualizar
docs = {
    "perro":      [0.9, 0.1, 0.1],
    "gato":       [0.8, 0.2, 0.1],
    "automóvil":  [0.1, 0.8, 0.2],
    "camión":     [0.1, 0.9, 0.1],
    "galaxia":    [0.2, 0.1, 0.9],
}
query = normalizar([1.0, 0.15, 0.1])   # Consulta: parecido a "perro"

print("Similitud coseno con query=[1.0, 0.15, 0.1]:")
for nombre, vec in sorted(docs.items(), key=lambda x: -similitud_coseno(normalizar(x[1]), query)):
    sim = similitud_coseno(normalizar(vec), query)
    print(f"  {nombre:<15} {sim:.4f}  {'█'*int(sim*20)}")

In [ ]:
# 🔨 IMPLEMENTACIÓN: VectorStore — Fuerza Bruta

class VectorStoreBruta:
    """
    Búsqueda exacta por fuerza bruta.
    Complejidad: O(n·d) por query — lineal en número de documentos.
    """
    def __init__(self, metrica='coseno'):
        self.vectores = {}    # id → vector
        self.metrica = metrica
    
    def agregar(self, doc_id, vector):
        self.vectores[doc_id] = list(vector)
    
    def buscar(self, query, k=5):
        """Retorna los k documentos más similares."""
        if self.metrica == 'coseno':
            scores = [(doc_id, similitud_coseno(query, vec))
                      for doc_id, vec in self.vectores.items()]
            scores.sort(key=lambda x: -x[1])
        else:
            scores = [(doc_id, distancia_euclidiana(query, vec))
                      for doc_id, vec in self.vectores.items()]
            scores.sort(key=lambda x: x[1])
        return scores[:k]

# Demo
store = VectorStoreBruta(metrica='coseno')
for nombre, vec in docs.items():
    store.agregar(nombre, normalizar(vec))

resultado = store.buscar(query, k=3)
print("Top-3 resultados (fuerza bruta):")
for doc_id, score in resultado:
    print(f"  {doc_id:<15} score={score:.4f}")

In [ ]:
# 🔨 IMPLEMENTACIÓN: KD-Tree (simplificado para dimensiones bajas)

class KDNode:
    def __init__(self, punto, doc_id, eje, izquierda=None, derecha=None):
        self.punto = punto
        self.doc_id = doc_id
        self.eje = eje
        self.izquierda = izquierda
        self.derecha = derecha

class KDTreeVectorStore:
    """
    KD-Tree para búsqueda aproximada eficiente en dimensiones bajas (d < 20).
    Complejidad: O(d·log n) promedio, O(n) peor caso.
    
    ⚠️ Para dimensiones altas (d > 20), el KD-Tree degenera (curse of dimensionality).
    Usar HNSW o LSH para embeddings de alta dimensión.
    """
    def __init__(self):
        self.raiz = None
        self._puntos = []   # (vector, doc_id) para reconstruir
    
    def agregar(self, doc_id, vector):
        self._puntos.append((vector, doc_id))
        self.raiz = self._construir(self._puntos[:], 0)
    
    def _construir(self, puntos, profundidad):
        if not puntos: return None
        eje = profundidad % len(puntos[0][0])
        puntos.sort(key=lambda p: p[0][eje])
        medio = len(puntos) // 2
        return KDNode(
            puntos[medio][0], puntos[medio][1], eje,
            self._construir(puntos[:medio], profundidad + 1),
            self._construir(puntos[medio+1:], profundidad + 1)
        )
    
    def buscar(self, query, k=5):
        import heapq
        resultado = []   # max-heap por distancia
        
        def _buscar(nodo):
            if nodo is None: return
            dist = distancia_euclidiana(query, nodo.punto)
            heapq.heappush(resultado, (-dist, nodo.doc_id))
            if len(resultado) > k:
                heapq.heappop(resultado)
            
            diff = query[nodo.eje] - nodo.punto[nodo.eje]
            primero = nodo.izquierda if diff <= 0 else nodo.derecha
            segundo = nodo.derecha  if diff <= 0 else nodo.izquierda
            _buscar(primero)
            
            if len(resultado) < k or abs(diff) < -resultado[0][0]:
                _buscar(segundo)
        
        _buscar(self.raiz)
        return sorted([(-d, doc_id) for d, doc_id in resultado])

kd = KDTreeVectorStore()
for nombre, vec in docs.items():
    kd.agregar(nombre, normalizar(vec))

print("Top-3 resultados (KD-Tree, distancia euclidiana):")
for dist, doc_id in kd.buscar(query, k=3):
    print(f"  {doc_id:<15} dist={dist:.4f}")

In [ ]:
# 🔨 IMPLEMENTACIÓN: LSH (Locality Sensitive Hashing)

import random

class LSHVectorStore:
    """
    Locality Sensitive Hashing: búsqueda aproximada O(1) promedio.
    
    Idea: vectores similares tienen alta probabilidad de tener el mismo hash.
    Usamos random hyperplane hashing: sign(w · v) para cada hiperplano aleatorio w.
    """
    def __init__(self, dim, num_tablas=5, num_bits=8, semilla=42):
        random.seed(semilla)
        self.num_tablas = num_tablas
        self.num_bits = num_bits
        # Generar hiperplanos aleatorios para cada tabla
        self.hiperplanos = [
            [[random.gauss(0,1) for _ in range(dim)] for _ in range(num_bits)]
            for _ in range(num_tablas)
        ]
        self.tablas = [{} for _ in range(num_tablas)]
        self.vectores = {}
    
    def _hash(self, tabla_idx, vector):
        bits = tuple(
            1 if sum(h[i]*v for i,v in enumerate(vector)) >= 0 else 0
            for h in self.hiperplanos[tabla_idx]
        )
        return bits
    
    def agregar(self, doc_id, vector):
        self.vectores[doc_id] = vector
        for i in range(self.num_tablas):
            h = self._hash(i, vector)
            self.tablas[i].setdefault(h, []).append(doc_id)
    
    def buscar(self, query, k=5):
        candidatos = set()
        for i in range(self.num_tablas):
            h = self._hash(i, query)
            candidatos.update(self.tablas[i].get(h, []))
        
        if not candidatos:
            candidatos = set(self.vectores.keys())
        
        scores = [(doc_id, similitud_coseno(query, self.vectores[doc_id]))
                  for doc_id in candidatos]
        scores.sort(key=lambda x: -x[1])
        return scores[:k]

# Demo con más vectores para ver el beneficio
random.seed(0)
dim = 10
lsh = LSHVectorStore(dim=dim, num_tablas=5, num_bits=8)

# Añadir 100 vectores aleatorios
for i in range(100):
    v = normalizar([random.gauss(0,1) for _ in range(dim)])
    lsh.agregar(f"doc-{i}", v)

# Añadir vectores "cercanos" para la query
query_alta_dim = normalizar([1.0]*dim)
lsh.agregar("doc-cercano-1", normalizar([0.9]*5 + [0.1]*5))
lsh.agregar("doc-cercano-2", normalizar([1.0]*8 + [0.2]*2))

resultado = lsh.buscar(query_alta_dim, k=5)
print("Top-5 LSH (similaridad coseno con query uniforme):")
for doc_id, score in resultado:
    print(f"  {doc_id:<20} score={score:.4f}")

## ✏️ Ejercicio 1: Búsqueda por Rango (Radio Search)

**Problema:** En lugar de los K más cercanos, retorna TODOS los documentos
dentro de un radio de similitud dado.

In [ ]:
# ✏️ EJERCICIO 1

def busqueda_radio(store: VectorStoreBruta, query, radio_coseno=0.9):
    """
    Retorna todos los documentos con similitud coseno >= radio_coseno.
    
    Args:
        store: VectorStoreBruta con documentos
        query: vector de consulta
        radio_coseno: umbral mínimo de similitud [0,1]
    
    Returns:
        list[(doc_id, score)] ordenados por similitud descendente
    """
    # TODO: Implementar
    pass

# Test
store_grande = VectorStoreBruta()
for nombre, vec in docs.items():
    store_grande.agregar(nombre, normalizar(vec))

resultado_radio = busqueda_radio(store_grande, query, radio_coseno=0.7)
print(f"Documentos con similitud >= 0.7:")
for doc_id, score in resultado_radio:
    print(f"  {doc_id}: {score:.4f}")

In [ ]:
# 💡 SOLUCIÓN — Ejercicio 1

def busqueda_radio(store: VectorStoreBruta, query, radio_coseno=0.9):
    resultados = [
        (doc_id, similitud_coseno(query, vec))
        for doc_id, vec in store.vectores.items()
        if similitud_coseno(query, vec) >= radio_coseno
    ]
    return sorted(resultados, key=lambda x: -x[1])

resultado_radio = busqueda_radio(store_grande, query, radio_coseno=0.7)
print(f"Documentos con similitud >= 0.7:")
for doc_id, score in resultado_radio:
    print(f"  {doc_id}: {score:.4f}")

## ✏️ Ejercicio 2: IVF — Inverted File Index

**Idea:** Divide el espacio vectorial en K clusters (K-Means).
Para una query, solo busca en los clusters más cercanos.

In [ ]:
# ✏️ EJERCICIO 2

class IVFVectorStore:
    """
    Inverted File Index con K-Means clustering.
    Reduce búsqueda exhaustiva a O(k·n/K) donde k = clusters a probar.
    """
    def __init__(self, num_clusters=3):
        self.num_clusters = num_clusters
        self.centroides = []
        self.clusters = {}     # cluster_id → [(vector, doc_id)]
        self.todos = []
    
    def agregar(self, doc_id, vector):
        self.todos.append((vector, doc_id))
    
    def construir_indice(self, iteraciones=10):
        """Ejecuta K-Means para asignar documentos a clusters."""
        # TODO: Implementar K-Means simplificado + asignación de documentos
        pass
    
    def buscar(self, query, k=5, clusters_a_probar=2):
        """Busca en los 'clusters_a_probar' clusters más cercanos a la query."""
        # TODO: Implementar búsqueda restringida a clusters más cercanos
        pass

# Test
ivf = IVFVectorStore(num_clusters=3)
for nombre, vec in docs.items():
    ivf.agregar(nombre, normalizar(vec))
# ivf.construir_indice()
# print(ivf.buscar(query, k=3))
print("IVF estructura lista para implementar")

In [ ]:
# 💡 SOLUCIÓN — Ejercicio 2

class IVFVectorStore:
    def __init__(self, num_clusters=3):
        self.num_clusters = num_clusters
        self.centroides = []
        self.clusters = {}
        self.todos = []
    
    def agregar(self, doc_id, vector):
        self.todos.append((vector, doc_id))
    
    def construir_indice(self, iteraciones=20, semilla=42):
        random.seed(semilla)
        dim = len(self.todos[0][0])
        # Inicializar centroides aleatorios
        self.centroides = [list(v) for v, _ in random.sample(self.todos, min(self.num_clusters, len(self.todos)))]
        
        for _ in range(iteraciones):
            self.clusters = {i: [] for i in range(len(self.centroides))}
            for vector, doc_id in self.todos:
                idx = min(range(len(self.centroides)),
                          key=lambda i: distancia_euclidiana(vector, self.centroides[i]))
                self.clusters[idx].append((vector, doc_id))
            
            # Recalcular centroides
            for i, miembros in self.clusters.items():
                if miembros:
                    self.centroides[i] = [
                        sum(m[0][d] for m in miembros) / len(miembros)
                        for d in range(dim)
                    ]
    
    def buscar(self, query, k=5, clusters_a_probar=2):
        if not self.centroides:
            self.construir_indice()
        
        # Encontrar clusters más cercanos
        distancias = sorted(enumerate(self.centroides),
                            key=lambda ic: distancia_euclidiana(query, ic[1]))
        
        candidatos = []
        for idx, _ in distancias[:clusters_a_probar]:
            candidatos.extend(self.clusters.get(idx, []))
        
        scores = [(doc_id, similitud_coseno(query, vec)) for vec, doc_id in candidatos]
        return sorted(scores, key=lambda x: -x[1])[:k]

ivf = IVFVectorStore(num_clusters=3)
for nombre, vec in docs.items():
    ivf.agregar(nombre, normalizar(vec))
ivf.construir_indice()

print("Top-3 IVF:")
for doc_id, score in ivf.buscar(query, k=3):
    print(f"  {doc_id}: {score:.4f}")

## 🎯 Quiz — Preguntas de Entrevista

**P1:** ¿Por qué el KD-Tree pierde eficiencia en dimensiones altas?
> **R:** "Curse of dimensionality": en altas dimensiones, todos los puntos están
> aproximadamente a la misma distancia entre sí. El KD-Tree explora casi todos los nodos.
> Para d > 20, la búsqueda lineal puede ser más rápida.

**P2:** ¿Qué es HNSW y por qué domina en benchmarks ANN?
> **R:** Hierarchical Navigable Small World — grafo de proximidad multicapa.
> Combina saltos largos (capas superiores) con búsqueda local fina (capa base).
> Logra >99% de recall con O(log n) complejidad y permite inserción incremental.

**P3:** ¿Cuál es la diferencia entre Pinecone y Weaviate?
> **R:** Pinecone es Vector DB as a Service (managed, simple API). Weaviate es open-source
> con esquema de objetos, filtros metatada y capacidad de almacenar el documento completo.
> Pinecone → simplicidad; Weaviate → control total y filtros complejos.

**P4:** En un RAG (Retrieval Augmented Generation), ¿qué determina la calidad de la búsqueda?
> **R:** La calidad del modelo de embedding (cómo representa el significado semántico),
> el tamaño del chunk (granularidad), el índice ANN (recall/precision), y el post-processing
> (re-ranking con un modelo cross-encoder más preciso pero costoso).